# Create, test, and serve prompt versions

[Read the guide](https://docs.futureagi.com/docs/cookbook/quickstart/prompt-versioning). Use a Python 3.12 notebook with Bash and venv support. You will create a new practice prompt, compare two versions on synthetic tickets, and roll back a custom serving label.

## Set up two Python environments

The platform SDK and Agent Learning Kit both install `fi/` modules. Keep them isolated. Only the test-case file crosses environments. Use an account where you can create a prompt.

In [ ]:
import os
import tempfile

# Use a fresh directory rather than overwriting an existing example.
workspace = tempfile.mkdtemp(prefix="ridgeline-")
os.chdir(workspace)
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["PIP_QUIET"] = "1"

In [ ]:
import os
from getpass import getpass

def require_secret(name):
    if os.environ.get(name):
        return
    value = getpass(f"{name} (hidden): ")
    if not value.strip():
        raise ValueError(f"{name} is required for this step")
    os.environ[name] = value.strip()

for name in ("FI_API_KEY", "FI_SECRET_KEY", "OPENAI_API_KEY"):
    require_secret(name)

In [ ]:
%%bash
set -euo pipefail
python3.12 -m venv .venv-platform
.venv-platform/bin/python -m pip install futureagi==0.8.0
python3.12 -m venv .venv-alk
.venv-alk/bin/python -m pip install agent-learning-kit==0.3.7
export PROMPT_NAME="ridgeline-ticket-router-$(python3.12 -c 'import uuid; print(uuid.uuid4().hex[:12])')"
export CANDIDATE_VERSION="v2"
# A notebook shell ends with the cell. Save only these two non-secret values.
printf '%s\n' "$PROMPT_NAME" "$CANDIDATE_VERSION" > router-versions.txt

In [ ]:
from pathlib import Path
os.environ["PROMPT_NAME"], os.environ["CANDIDATE_VERSION"] = Path("router-versions.txt").read_text().splitlines()

## 1. Commit a baseline and a candidate

In [ ]:
%%writefile create_versions.py
import os
from fi.prompt import Prompt
from fi.prompt.types import ModelConfig, PromptTemplate, SystemMessage, UserMessage

name = os.environ["PROMPT_NAME"]
routing = (
    "Route tickets for Ridgeline Outfitters. Unused gear goes to returns. "
    "Damaged gear goes to repairs. "
)

def template(instruction):
    return PromptTemplate(
        name=name,
        messages=[
            SystemMessage(content=routing + instruction),
            UserMessage(content="Ticket: {{ticket}}"),
        ],
        model_configuration=ModelConfig(model_name="gpt-4o-mini", temperature=0),
    )

if __name__ == "__main__":
    prompt = Prompt(template=template("Recommend the right team in a sentence."))
    prompt.create()
    if not prompt.get_label_by_name("ridgeline-live"):
        prompt.create_label("ridgeline-live")
    prompt.commit_current_version(message="Route tickets for a support rep", label="ridgeline-live")
    prompt.create_new_version(template=template("Return only the queue code: RETURNS or REPAIRS."))
    prompt.commit_current_version(message="Return a code the ticket system can dispatch")
    live = Prompt.get_template_by_name(name=name, label="ridgeline-live")
    if live.template.version != "v1" or prompt.template.version != "v2":
        raise RuntimeError("Check the baseline label and candidate before exporting")
    print("ridgeline-live:", live.template.version, "candidate:", prompt.template.version)

In [ ]:
%%bash
set -euo pipefail
.venv-platform/bin/python create_versions.py

## 2. Export the same tickets for both versions

In [ ]:
%%writefile render_router.py
def render_router(prompt, ticket):
    messages = []
    for message in prompt.template.model_dump()["messages"]:
        content = message["content"]
        if isinstance(content, list):
            if any(part.get("type") != "text" for part in content):
                raise ValueError("This router expects text parts only")
            content = "".join(part["text"] for part in content)
        messages.append({"role": message["role"], "content": content.replace("{{ticket}}", ticket)})
    return messages

In [ ]:
%%writefile export_cases.py
import json
import os
from pathlib import Path
from fi.prompt import Prompt
from render_router import render_router

cases = [
    ("The unused pack is too small. Can I return it?", "RETURNS"),
    ("The seam on my tent tore. Can you fix it?", "REPAIRS"),
]
rows = []
for version in ("v1", os.environ["CANDIDATE_VERSION"]):
    prompt = Prompt.get_template_by_name(name=os.environ["PROMPT_NAME"], version=version)
    for ticket, expected in cases:
        rows.append({"version": version, "expected": expected,
                     "messages": render_router(prompt, ticket)})
Path("router-cases.json").write_text(json.dumps(rows, indent=2))
print(f"Exported {len(rows)} cases")

In [ ]:
%%bash
set -euo pipefail
.venv-platform/bin/python export_cases.py

## 3. Score the replies before promotion

Each reply must match the expected queue code, including case. Save the printed scores so promotion can check both candidate tickets.

In [ ]:
%%writefile score_cases.py
import json
from pathlib import Path
from fi.evals import evaluate
from openai import OpenAI

client = OpenAI()

for row in json.loads(Path("router-cases.json").read_text()):
    reply = client.chat.completions.create(
        model="gpt-4o-mini", messages=row["messages"], temperature=0
    )
    text = reply.choices[0].message.content
    if not isinstance(text, str):
        raise RuntimeError("Model returned no text to score")
    result = evaluate("equals", engine="local", output=text,
                      expected_response=row["expected"], config={"case_sensitive": True})
    if result.error:
        raise RuntimeError(result.error)
    print(row["version"], row["expected"], "PASS" if result.passed else "FAIL", repr(text))

In [ ]:
%%bash
set -euo pipefail
.venv-alk/bin/python score_cases.py | tee router-scores.txt

## 4. Promote, serve, or roll back

Only promote if both candidate replies match exactly. If either fails, stop here and review the prompt before trying again.

In [ ]:
%%writefile serve.py
import os
import sys
from fi.prompt import Prompt
from render_router import render_router

name = os.environ["PROMPT_NAME"]
if len(sys.argv) > 2:
    raise SystemExit("Usage: serve.py [version]")
if len(sys.argv) == 2:
    Prompt.assign_label_to_template_version(template_name=name, version=sys.argv[1],
                                            label="ridgeline-live")
live = Prompt.get_template_by_name(name=name, label="ridgeline-live")
if len(sys.argv) == 2 and live.template.version != sys.argv[1]:
    raise SystemExit(f"Label resolved to {live.template.version}, not {sys.argv[1]}")
messages = render_router(live, "The seam on my tent tore. Can you fix it?")
print("Serving", live.template.version)
print(messages[-1]["content"])

In [ ]:
%%bash
set -euo pipefail
grep -Fqx "$CANDIDATE_VERSION RETURNS PASS 'RETURNS'" router-scores.txt
grep -Fqx "$CANDIDATE_VERSION REPAIRS PASS 'REPAIRS'" router-scores.txt
.venv-platform/bin/python serve.py "${CANDIDATE_VERSION:?Set the tested version first}"
.venv-platform/bin/python serve.py

Move the label back to v1. This does not delete either version.

In [ ]:
%%bash
set -euo pipefail
.venv-platform/bin/python serve.py v1